# VGG16

## Overview

VGG16 was proposed by the Visual Geometry Group (VGG) at the University of Oxford in 2014. It demonstrated that increasing network depth using small convolution filters significantly improves image classification performance.

## Key Features

- Uses only 3×3 convolution filters.
- Simple and uniform architecture.
- Consists of 16 learnable layers.
- Produces highly discriminative image features.

## Architecture Summary

Input Image
→ Multiple 3×3 Convolution Layers
→ Max Pooling
→ Multiple 3×3 Convolution Layers
→ Max Pooling
→ Fully Connected Layers
→ Output Layer

## Applications

- Image classification
- Transfer learning
- Feature extraction
- Object detection

## Limitations

- Very large model size.
- High memory consumption.
- Slow training and inference compared to newer architectures.

In [11]:
import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Dropout, Flatten
from tensorflow.keras.preprocessing.image import ImageDataGenerator

import matplotlib.pyplot as plt
import sklearn

import warnings
warnings.filterwarnings('ignore')

In [12]:
train_dir = "HAR3Class/train"
test_dir = "HAR3Class/test"

train_datagen = ImageDataGenerator(
    rescale=1.0/255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True)

test_datagen = ImageDataGenerator(rescale = 1.0/255)

train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size = (224, 224),
    batch_size = 32, # no of images to be read at once
    class_mode = 'categorical'
) ## assigns labels based on folder names.


test_data = test_datagen.flow_from_directory(
    test_dir,
    target_size = (224, 224),
    batch_size = 32, # no of images to be read at once
    class_mode = 'categorical'
) 

## here if i put target_size = (227,227), then it will automatically resize the images, but not pad it

Found 610 images belonging to 3 classes.


Found 150 images belonging to 3 classes.


In [13]:
from keras import layers
def create_vgg16(input_shape, num_classes):
    model = Sequential()

    # Block 1
    model.add(Conv2D(64, kernel_size = (3,3), strides = (1,1), activation = 'relu', input_shape=input_shape, padding = 'same'))
    model.add(Conv2D(64, kernel_size = (3,3), strides = (1,1), activation = 'relu', padding = 'same'))
    model.add(MaxPooling2D(pool_size = (2,2), strides = (2,2)))

    # Block 2
    model.add(Conv2D(128, kernel_size = (3,3), strides = (1,1), activation = 'relu', padding = 'same'))
    model.add(Conv2D(128, kernel_size = (3,3), strides = (1,1), activation = 'relu', padding = 'same'))
    model.add(MaxPooling2D(pool_size = (2,2), strides = (2,2)))
    
    # Block 3
    model.add(Conv2D(256, kernel_size = (3,3), strides = (1,1), activation = 'relu', padding = 'same'))
    model.add(Conv2D(256, kernel_size = (3,3), strides = (1,1), activation = 'relu', padding = 'same'))
    model.add(Conv2D(256, kernel_size = (3,3), strides = (1,1), activation = 'relu', padding = 'same'))
    model.add(MaxPooling2D(pool_size = (2,2), strides = (2,2)))
    
    # Block 4
    model.add(Conv2D(512, kernel_size = (3,3), strides = (1,1), activation = 'relu', padding = 'same'))
    model.add(Conv2D(512, kernel_size = (3,3), strides = (1,1), activation = 'relu', padding = 'same'))
    model.add(Conv2D(512, kernel_size = (3,3), strides = (1,1), activation = 'relu', padding = 'same'))
    model.add(MaxPooling2D(pool_size = (2,2), strides = (2,2)))
    
    # Block 5
    model.add(Conv2D(512, kernel_size = (3,3), strides = (1,1), activation = 'relu', padding = 'same'))
    model.add(Conv2D(512, kernel_size = (3,3), strides = (1,1), activation = 'relu', padding = 'same'))
    model.add(Conv2D(512, kernel_size = (3,3), strides = (1,1), activation = 'relu', padding = 'same'))
    model.add(MaxPooling2D(pool_size = (2,2), strides = (2,2)))

    
    # flattening layer and fully connected layers 
    model.add(Flatten())
    model.add(Dense(4096, activation ='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(4096, activation ='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(num_classes, activation = 'softmax'))

    return model    

## padding takes only 2 values: valid / same. valid = no padding (output shrinks) and same = adds 0 around the edges (output same as input)

In [14]:
# Define the model input shape and number of output classes (3 classes in HAR3class)
input_shape = (224, 224, 3)
num_classes = 3

# Create the AlexNet model
vgg16 = create_vgg16(input_shape, num_classes)

# Display model summary
vgg16.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_13 (Conv2D)                   │ (None, 224, 224, 64)        │           1,792 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_14 (Conv2D)                   │ (None, 224, 224, 64)        │          36,928 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_5 (MaxPooling2D)       │ (None, 112, 112, 64)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_15 (Conv2D)                   │ (None, 112, 112, 128)       │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_16 (Conv2D)                   │ (None, 112, 112, 128)       │         147,584 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_6 (MaxPooling2D)       │ (None, 56, 56, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_17 (Conv2D)                   │ (None, 56, 56, 256)         │         295,168 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_18 (Conv2D)                   │ (None, 56, 56, 256)         │         590,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_19 (Conv2D)                   │ (None, 56, 56, 256)         │         590,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_7 (MaxPooling2D)       │ (None, 28, 28, 256)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_20 (Conv2D)                   │ (None, 28, 28, 512)         │       1,180,160 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_21 (Conv2D)                   │ (None, 28, 28, 512)         │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_22 (Conv2D)                   │ (None, 28, 28, 512)         │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_8 (MaxPooling2D)       │ (None, 14, 14, 512)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_23 (Conv2D)                   │ (None, 14, 14, 512)         │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_24 (Conv2D)                   │ (None, 14, 14, 512)         │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_25 (Conv2D)                   │ (None, 14, 14, 512)         │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_9 (MaxPooling2D)       │ (None, 7, 7, 512)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_1 (Flatten)                  │ (None, 25088)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 4096)                │     102,764,544 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 4096)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼──────────────

 Total params: 134,272,835 (512.21 MB)

 Trainable params: 134,272,835 (512.21 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
# Compiling the model
vgg16.compile(
    loss = 'categorical_crossentropy',
    optimizer = 'sgd',
    metrics = ['accuracy']
)

history = vgg16.fit(train_data, validation_data = test_data, epochs=5)

## sparse categorical crossentropy is for 0,1,2,3....
## categorical cross entropy is for one hot encoded variables

Epoch 1/5


 1/20 ━━━━━━━━━━━━━━━━━━━━ 5:16 17s/step - accuracy: 0.4062 - loss: 1.0960

 2/20 ━━━━━━━━━━━━━━━━━━━━ 7:12 24s/step - accuracy: 0.3906 - loss: 1.0963

 3/20 ━━━━━━━━━━━━━━━━━━━━ 3:35 13s/step - accuracy: 0.3816 - loss: 1.0966

 4/20 ━━━━━━━━━━━━━━━━━━━━ 3:30 13s/step - accuracy: 0.3883 - loss: 1.0966

...

17/20 ━━━━━━━━━━━━━━━━━━━━ 40s 14s/step - accuracy: 0.3450 - loss: 1.0984

18/20 ━━━━━━━━━━━━━━━━━━━━ 27s 14s/step - accuracy: 0.3455 - loss: 1.0984

19/20 ━━━━━━━━━━━━━━━━━━━━ 13s 14s/step - accuracy: 0.3457 - loss: 1.0984

20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13s/step - accuracy: 0.3457 - loss: 1.0984 

20/20 ━━━━━━━━━━━━━━━━━━━━ 286s 14s/step - accuracy: 0.3459 - loss: 1.0985 - val_accuracy: 0.3333 - val_loss: 1.0987


In [ ]:
pd.DataFrame(history.history).plot()

In [ ]:
model.evaluate(X_test, y_test)